# Practice: curvature, learning rates and Newton's method
*AI from scratch, calculus 3* · lesson: https://ihor.sh/ai/c3-taylor-and-curvature.html

Taylor approximations, the learning-rate limit 2/k found by experiment, a Hessian from PyTorch, the condition
number of a real dataset before and after scaling, and Newton's method racing gradient descent.
Fill in each `...` under `# YOUR CODE HERE`, then run the check under it (✓ = right, ✗ = not yet).

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['Sum terms x^k / math.factorial(k) for k from 0 through n inclusive.'], 2: ['Compute absolute error |taylor_exp(1.0, n) - math.e| for each n in range(1, 9).'], 3: ['Write a helper simulating 100 GD steps: x = x - eta * k * x starting from x = 1.0.', 'Filter `rates` keeping only those where |x| after 100 steps is < 1.0.'], 4: ['Compute Hessian with torch.autograd.functional.hessian(f, torch.tensor([1.0, 1.0])).numpy().', 'Compute eigenvalues using np.linalg.eigvalsh(H) (returns ascending sorted eigenvalues).', 'Compute condition number cond = evals[-1] / evals[0].'], 5: ['Build Xr by stacking raw features D with an intercept column of 1s.', 'Build Xs by standardizing D (subtract mean, divide by std) then stacking with 1s.', 'Compute condition numbers np.linalg.cond(Xr.T @ Xr) and np.linalg.cond(Xs.T @ Xs).'], 6: ['Initialize positions gd = 0.0 and nt = 0.0, and error lists gd_err = [], nt_err = [].', "For 8 steps: update gd with gradient descent, nt with Newton's method.", 'Append absolute errors abs(x - math.log(2)) to the corresponding lists.'], 7: ['Compute eigenvalues using ev = np.linalg.eigvalsh(H).', 'If all eigenvalues are positive ((ev > 0).all()), return "minimum".', 'If all eigenvalues are negative ((ev < 0).all()), return "maximum".', 'Otherwise, eigenvalues have mixed signs or zero; return "saddle".']}
_ANSWERS = {1: 'cmV0dXJuIHN1bSh4ICoqIGsgLyBtYXRoLmZhY3RvcmlhbChrKSBmb3IgayBpbiByYW5nZShuICsgMSkp', 2: 'ZXJyb3JzID0gW2Ficyh0YXlsb3JfZXhwKDEuMCwgbikgLSBtYXRoLmUpIGZvciBuIGluIHJhbmdlKDEsIDkpXQ==', 3: 'ZGVmIHJ1bihldGEsIHg9MS4wKToKICAgIGZvciBfIGluIHJhbmdlKDEwMCk6CiAgICAgICAgeCAtPSBldGEgKiBrICogeAogICAgcmV0dXJuIHgKc3RhYmxlID0gW2V0YSBmb3IgZXRhIGluIHJhdGVzIGlmIGFicyhydW4oZXRhKSkgPCAxXQ==', 4: 'SCA9IHRvcmNoLmF1dG9ncmFkLmZ1bmN0aW9uYWwuaGVzc2lhbihmLCB0b3JjaC50ZW5zb3IoWzEuMCwgMS4wXSkpLm51bXB5KCkKZXZhbHMgPSBucC5saW5hbGcuZWlndmFsc2goSCkKY29uZCA9IGV2YWxzWy0xXSAvIGV2YWxzWzBd', 5: 'WHIgPSBucC5jb2x1bW5fc3RhY2soW0QsIG5wLm9uZXMobGVuKEQpKV0pClhzID0gbnAuY29sdW1uX3N0YWNrKFsoRCAtIEQubWVhbigwKSkgLyBELnN0ZCgwKSwgbnAub25lcyhsZW4oRCkpXSkKY29uZF9yYXcgPSBucC5saW5hbGcuY29uZChYci5UIEAgWHIpCmNvbmRfc3RkID0gbnAubGluYWxnLmNvbmQoWHMuVCBAIFhzKQ==', 6: 'Z2QgPSBudCA9IDAuMDsgZ2RfZXJyLCBudF9lcnIgPSBbXSwgW10KZm9yIF8gaW4gcmFuZ2UoOCk6CiAgICBnZCAtPSAwLjMgKiBkMShnZCk7IG50IC09IGQxKG50KSAvIGQyKG50KQogICAgZ2RfZXJyLmFwcGVuZChhYnMoZ2QgLSBtYXRoLmxvZygyKSkpOyBudF9lcnIuYXBwZW5kKGFicyhudCAtIG1hdGgubG9nKDIpKSk=', 7: 'ZXYgPSBucC5saW5hbGcuZWlndmFsc2goSCkKcmV0dXJuICJtaW5pbXVtIiBpZiAoZXYgPiAwKS5hbGwoKSBlbHNlICJtYXhpbXVtIiBpZiAoZXYgPCAwKS5hbGwoKSBlbHNlICJzYWRkbGUi'}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch

## 1. Taylor polynomials
$e^x \approx \sum_{k=0}^{n} x^k / k!$. Write `taylor_exp(x, n)` using `factorial` (`math.factorial`), and set `errors`: the absolute error at x = 1 for
n = 1 … 8. Each extra term should win roughly another digit.

In [ ]:
def taylor_exp(x, n):
    """Compute the n-th order Taylor polynomial approximation of e^x around 0.
    
    Formula: sum_{k=0}^n (x^k / k!)
    
    Args:
        x: Float evaluation point.
        n: Integer polynomial degree (highest power of x).
    Returns:
        Float approximation of e^x.
    """
    # YOUR CODE HERE (stuck? hint(1), then answer(1))
    ...

errors = None
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...
print(errors)

In [ ]:
check(2, lambda: np.isclose(taylor_exp(0.5, 12), math.exp(0.5)), 
      "taylor_exp(0.5, 12) should closely approximate math.exp(0.5).")
check(2, lambda: errors[-1] < 1e-4 and all(a > b for a, b in zip(errors, errors[1:])), 
      "The approximation error should shrink monotonically with each additional term.")

## 2. The learning-rate limit, by experiment
On $f(x) = \frac{k}{2}x^2$ (curvature k = 8), gradient descent multiplies x by $(1 - \eta k)$ each step.
Using a simulation helper `run(eta)`, run 100 steps from x = 1 for each learning rate in `rates` and set `stable` to the rates where |x| ends below 1.
The lesson says the limit is $2/k = 0.25$.

In [ ]:
k = 8.0
rates = [0.05, 0.1, 0.2, 0.24, 0.26, 0.3]
stable = None
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...
print(stable)

In [ ]:
check(3, lambda: stable == [0.05, 0.1, 0.2, 0.24], 
      "Rates strictly below 2/k = 0.25 should remain stable ([0.05, 0.1, 0.2, 0.24]), while 0.26 and 0.3 explode.")

## 3. A Hessian from PyTorch
For $f(x, y) = 3x^2 + 2xy + y^2$, get the Hessian at (1, 1) with `hessian` (`torch.autograd.functional.hessian`, set `H`),
its eigenvalues (`evals`, ascending) and the condition number `cond` = largest / smallest.

In [ ]:
f = lambda v: 3 * v[0] ** 2 + 2 * v[0] * v[1] + v[1] ** 2
H = evals = cond = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
print(H, evals, cond)

In [ ]:
check(4, lambda: np.allclose(H, [[6, 2], [2, 2]]), 
      "The Hessian matrix H should be [[6, 2], [2, 2]].")
check(4, lambda: np.isclose(cond, (4 + 2 * np.sqrt(2)) / (4 - 2 * np.sqrt(2))), 
      "Condition number is lambda_max / lambda_min (about 5.83).")

## 4. Scaling a real dataset un-stretches the valley
For least squares the Hessian is $\frac{2}{n} X^\top X$. Compute the condition number of $X^\top X$ for the
diabetes features **raw** (with an intercept column of 1s) and **standardized** (z-scores, then the 1s).
Set `cond_raw` and `cond_std`.

In [ ]:
from sklearn.datasets import load_diabetes
D = load_diabetes(scaled=False).data
cond_raw = cond_std = None
# YOUR CODE HERE (stuck? hint(5), then answer(5))
...
print("raw:", cond_raw, "  standardized:", cond_std)

In [ ]:
check(5, lambda: cond_raw > 1e6 and cond_std < 1e3, 
      "Raw condition number should exceed 1e6, while standardized should drop below 1e3.")

## 5. Newton vs gradient descent
Minimize $f(x) = e^x - 2x$ (minimum at $\ln 2$) from x = 0. Gradient descent: `x -= 0.3 * f'(x)`.
Newton: `x -= f'(x) / f''(x)`. Record each method's error after every step for 8 steps in `gd_err` and `nt_err`.

In [ ]:
d1, d2 = lambda x: math.exp(x) - 2, lambda x: math.exp(x)
gd_err = nt_err = None
# YOUR CODE HERE (stuck? hint(6), then answer(6))
...
if nt_err is not None and len(nt_err):
    plt.semilogy(gd_err, "o-", label="gradient descent"); plt.semilogy(np.maximum(nt_err, 1e-17), "o-", label="Newton"); plt.legend(); plt.show()

In [ ]:
check(6, lambda: nt_err[4] < 1e-12 and gd_err[4] > 1e-3, 
      "Newton's method should converge quadratically (error < 1e-12 by step 4) while GD converges linearly.")

## 6. Minimum, maximum or saddle?
Write `kind(H)`: "minimum" if every eigenvalue of the symmetric matrix H is positive, "maximum" if all are
negative, otherwise "saddle".

In [ ]:
def kind(H):
    """Classify critical point curvature from symmetric Hessian matrix H.
    
    Args:
        H: 2D square symmetric NumPy array (the Hessian matrix).
    Returns:
        str: "minimum" if all eigenvalues > 0, "maximum" if all < 0, else "saddle".
    """
    # YOUR CODE HERE (stuck? hint(7), then answer(7))
    ...

In [ ]:
check(7, lambda: [kind(np.diag([3, 1])), kind(np.diag([-1, -2])), kind(np.array([[1, 2], [2, 1]]))] == ["minimum", "maximum", "saddle"], 
      "Check kind() on positive-definite, negative-definite, and indefinite matrices.")

## Next
Back to the path: https://ihor.sh/ai/ · next: chapter 5, probability and loss.